# 🧠 Ejercicios: Anthropic
# Clientes API - sectores, planes, gasto y satisfacción

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de Anthropic.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué sector gasta más
### en la API y con qué
### ticket por cliente?

Gasto total y medio por sector.
Dónde está la cuenta grande.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'anthropic_clientes.csv'
)

r = (
  df
  .groupby('sector')
  .agg(
    total=('gasto_mensual_usd',
           'sum'),
    medio=('gasto_mensual_usd',
           'mean'),
    n=('gasto_mensual_usd',
       'size'),
  )
  .round(0)
  .sort_values('total',
    ascending=False)
)

print('Gasto por sector:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿El 20% de clientes
### genera el 80% del
### gasto? (Pareto)

Concentración de ingresos en las
cuentas top. Riesgo de depender.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'anthropic_clientes.csv'
)

s = (
  df['gasto_mensual_usd']
  .sort_values(ascending=False)
  .reset_index(drop=True)
)

top = int(len(s) * 0.2)
pct = (
  s.head(top).sum()
  / s.sum() * 100
)

print(f'Clientes: {len(s)}')
print(f'Top 20%: {top}')
print(
  f'Generan: {pct:.1f}% '
  'del gasto'
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué caso de uso sale
### más caro por millón
### de tokens?

Gasto / tokens x 1M. El coste real
normalizado por consumo.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'anthropic_clientes.csv'
)

g = (
  df
  .groupby('caso_uso')
  [['gasto_mensual_usd',
    'tokens_mes']]
  .sum()
)

g['usd_mtok'] = (
  g['gasto_mensual_usd']
  / g['tokens_mes'] * 1e6
).round(2)

print('Coste por millón tokens:')
print(
  g['usd_mtok']
  .sort_values(ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Pagar más plan sube
### la satisfacción del
### cliente?

Satisfacción media por plan y su
gasto. ¿El premium contenta más?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'anthropic_clientes.csv'
)

orden = ['Build', 'Scale',
  'Enterprise']

r = (
  df
  .groupby('plan')
  .agg(
    satis=('satisfaccion',
           'mean'),
    gasto=('gasto_mensual_usd',
           'mean'),
    n=('satisfaccion', 'size'),
  )
  .round(2)
  .reindex(orden)
)

print('Satisfacción por plan:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué clientes están
### en riesgo: alto gasto
### y baja satisfacción?

Cruce gasto alto + satisfacción
baja. Las cuentas a blindar ya.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'anthropic_clientes.csv'
)

umbral = (
  df['gasto_mensual_usd']
  .quantile(.75)
)

riesgo = df[
  (df['gasto_mensual_usd']
   >= umbral)
  & (df['satisfaccion'] < 3)
]

fuga = (
  riesgo['gasto_mensual_usd']
  .sum()
)

print(f'En riesgo: {len(riesgo)}')
print(f'Gasto en juego: {fuga:.0f}$')
print('\nPor sector:')
print(
  riesgo['sector']
  .value_counts()
)